# Agent 7 — Extracurricular Agent

This notebook builds Agent 7: it scores a candidate's non-academic profile — leadership,
research/publications, competitions, volunteering — into a single `profile_strength_score`,
and includes the SOP/LOR evaluation sub-module described in the framework doc. The output
feeds back into Agent 2's `RandomForestClassifier` as a feature (`extracurricular_score`)
and into Agent 5's merit-based scholarship eligibility check.

**Key design decisions (and why):**
- Like Agent 1's `profile_score`, a naive linear rubric total is *not* used as the sole
  feature set for the ML layer below — we keep the raw category scores (`leadership_score`,
  `research_score`, `competition_score`, `volunteering_score`, `publication_count`,
  `patent_count`) as features and treat `profile_strength_score` as the label the model
  learns to reproduce/generalize, the same leakage-avoidance pattern as Agent 1.
- Two disjoint evidence types exist per candidate — quantifiable achievements (counts,
  durations) and free-text evidence (descriptions, SOP) — so, mirroring Agent 1's
  MS/MBA `track` indicator, we use a `has_research_evidence` indicator rather than
  training separate models per candidate type.
- The SOP/LOR sub-module is intentionally **not** a fine-tuned model: it triangulates a
  rule-based structural score, an embedding-similarity score, and one Groq LLaMA rubric
  call — three cheap, reuse-existing-infra signals, as specified in the framework doc,
  rather than a fourth ML model trained on a currently-nonexistent labeled SOP dataset.
- Same reasoning as Agent 1's model-comparison step: we don't assume any one model family
  is right — Logistic Regression, Random Forest, Gradient Boosting, XGBoost, and SVM are
  compared head-to-head on the same split before a final model is picked.


## 0. Synthetic training data

Agent 7 has no existing labeled dataset (unlike Agent 1's `agent1_ms_augmented_v2.csv`), so this section generates a synthetic candidate-achievement dataset with a known ground-truth rubric, in the same spirit as Agent 1's `recommended_tier` threshold rule on `profile_score`. Swap `DATA_PATH` for real data the moment it exists — everything downstream only assumes the same column names.

In [1]:

import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)
N = 4000

def gen_synthetic_extracurricular_data(n=N, rng=RNG):
    leadership_score = np.clip(rng.normal(5, 2.3, n), 0, 10)
    research_score = np.clip(rng.normal(4, 2.6, n), 0, 10)
    competition_score = np.clip(rng.normal(3.5, 2.4, n), 0, 10)
    volunteering_score = np.clip(rng.normal(4.5, 2.2, n), 0, 10)

    publication_count = rng.poisson(0.6, n)
    patent_count = rng.poisson(0.08, n)
    has_research_evidence = ((publication_count > 0) | (patent_count > 0)).astype(int)

    degree_level = rng.choice(["Bachelors", "Masters", "PhD-applicant"], size=n, p=[0.55, 0.35, 0.10])
    field = rng.choice(["STEM", "Business", "SocialSci", "Arts", "Other"], size=n,
                        p=[0.45, 0.22, 0.15, 0.10, 0.08])

    # Weighted rubric (the framework's "weighted rubric: leadership, volunteering,
    # competitions, publications, patents") + noise -> ground-truth composite.
    composite = (
        0.28 * leadership_score
        + 0.24 * research_score
        + 0.16 * competition_score
        + 0.14 * volunteering_score
        + 6.0 * np.log1p(publication_count)
        + 10.0 * np.log1p(patent_count)
        + rng.normal(0, 3.0, n)
    )
    profile_strength_score = np.clip(composite / composite.max() * 100, 0, 100)
    strength_tier = np.where(profile_strength_score >= 65, "Strong",
                     np.where(profile_strength_score >= 40, "Moderate", "Weak"))

    df = pd.DataFrame({
        "candidate_id": [f"C{i:05d}" for i in range(n)],
        "degree_level": degree_level,
        "field": field,
        "leadership_score": leadership_score.round(2),
        "research_score": research_score.round(2),
        "competition_score": competition_score.round(2),
        "volunteering_score": volunteering_score.round(2),
        "publication_count": publication_count,
        "patent_count": patent_count,
        "has_research_evidence": has_research_evidence,
        "profile_strength_score": profile_strength_score.round(2),
        "strength_tier": strength_tier,
    })
    return df

DATA_PATH = "agent7_extracurricular_synthetic.csv"
df = gen_synthetic_extracurricular_data()
df.to_csv(DATA_PATH, index=False)
print("Shape:", df.shape)
df.head()


Shape: (4000, 12)


,candidate_id,degree_level,field,leadership_score,research_score,competition_score,volunteering_score,publication_count,patent_count,has_research_evidence,profile_strength_score,strength_tier
0,C00000,Masters,SocialSci,5.70,4.66,4.30,3.82,1,1,1,62.30,Moderate
1,C00001,Masters,STEM,2.61,6.33,6.47,4.63,0,0,0,30.77,Weak
2,C00002,Masters,Business,6.73,4.71,6.42,1.77,0,0,0,18.77,Weak
3,C00003,Bachelors,Business,7.16,9.82,2.42,1.45,1,0,1,35.33,Weak
4,C00004,Bachelors,SocialSci,0.51,7.72,3.97,4.90,1,0,1,25.59,Weak


## 1. Load and explore the data

In [2]:

df = pd.read_csv(DATA_PATH)
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())


candidate_id                  str
degree_level                  str
field                         str
leadership_score          float64
research_score            float64
competition_score         float64
volunteering_score        float64
publication_count           int64
patent_count                int64
has_research_evidence       int64
profile_strength_score    float64
strength_tier                 str
dtype: object

Missing values per column:
candidate_id              0
degree_level              0
field                     0
leadership_score          0
research_score            0
competition_score         0
volunteering_score        0
publication_count         0
patent_count              0
has_research_evidence     0
profile_strength_score    0
strength_tier             0
dtype: int64


In [3]:

print("Degree level distribution:")
print(df["degree_level"].value_counts())
print()
print("Strength tier distribution:")
print(df["strength_tier"].value_counts())
print()
print("Tier distribution by degree level:")
print(df.groupby("degree_level")["strength_tier"].value_counts())


Degree level distribution:
degree_level
Bachelors        2196
Masters          1411
PhD-applicant     393
Name: count, dtype: int64

Strength tier distribution:
strength_tier
Weak        2843
Moderate     983
Strong       174
Name: count, dtype: int64

Tier distribution by degree level:
degree_level   strength_tier
Bachelors      Weak             1560
               Moderate          542
               Strong             94
Masters        Weak              993
               Moderate          352
               Strong             66
PhD-applicant  Weak              290
               Moderate           89
               Strong             14
Name: count, dtype: int64


### Sanity check: confirm `profile_strength_score` is a leakage risk if kept raw

Like Agent 1's `profile_score`, `profile_strength_score` is a near-deterministic weighted sum of the category scores below it, and `strength_tier` is a threshold rule on it (~65 / ~40). It is used only as the training **label**, never as a feature.

In [4]:

corr_cols = ["leadership_score", "research_score", "competition_score",
             "volunteering_score", "publication_count", "patent_count",
             "profile_strength_score"]
print(df[corr_cols].corr()["profile_strength_score"])


leadership_score          0.125102
research_score            0.120482
competition_score         0.076898
volunteering_score        0.069241
publication_count         0.568162
patent_count              0.404532
profile_strength_score    1.000000
Name: profile_strength_score, dtype: float64


## 2. Preprocessing

In [5]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import joblib
import time

pd.set_option("display.max_columns", None)

model_df = df.drop(columns=["candidate_id", "profile_strength_score"])

# Target: predict strength_tier (3-class); we also keep the regression target
# (profile_strength_score) for a downstream regressor used to produce the
# continuous score the other agents actually consume.
y = model_df["strength_tier"]
X = model_df.drop(columns=["strength_tier"])

numeric_cols = ["leadership_score", "research_score", "competition_score",
                 "volunteering_score", "publication_count", "patent_count",
                 "has_research_evidence"]
cat_cols = ["degree_level", "field"]

for c in numeric_cols:
    X[c] = X[c].fillna(0)
for c in cat_cols:
    X[c] = X[c].fillna("NA")

X_encoded = pd.get_dummies(X, columns=cat_cols)
print(X_encoded.shape)
X_encoded.head()


(4000, 15)


,leadership_score,research_score,competition_score,volunteering_score,publication_count,patent_count,has_research_evidence,degree_level_Bachelors,degree_level_Masters,degree_level_PhD-applicant,field_Arts,field_Business,field_Other,field_STEM,field_SocialSci
0,5.70,4.66,4.30,3.82,1,1,1,False,True,False,False,False,False,False,True
1,2.61,6.33,6.47,4.63,0,0,0,False,True,False,False,False,False,True,False
2,6.73,4.71,6.42,1.77,0,0,0,False,True,False,False,True,False,False,False
3,7.16,9.82,2.42,1.45,1,0,1,True,False,False,False,True,False,False,False
4,0.51,7.72,3.97,4.90,1,0,1,True,False,False,False,False,False,False,True


In [6]:

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape, " Test size:", X_test.shape)


Train size: (3200, 15)  Test size: (800, 15)


## 3. Model comparison

Same approach as Agent 1: compare several model families on the identical split rather than assuming one is best.

In [7]:

candidate_models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000), True),
    "Random Forest": (RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1), False),
    "Gradient Boosting (sklearn)": (GradientBoostingClassifier(random_state=42), False),
    "XGBoost": (xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=42, eval_metric="mlogloss"), False),
    "SVM (RBF)": (SVC(probability=True, random_state=42), True),
}

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

y_train_enc = y_train.map({"Weak": 0, "Moderate": 1, "Strong": 2})
y_test_enc = y_test.map({"Weak": 0, "Moderate": 1, "Strong": 2})

comparison_results = []
fitted_models = {}
for name, (model, needs_scaling) in candidate_models.items():
    Xtr = X_train_s if needs_scaling else X_train
    Xte = X_test_s if needs_scaling else X_test
    ytr = y_train_enc if name == "XGBoost" else y_train

    t0 = time.time()
    model.fit(Xtr, ytr)
    train_time = time.time() - t0

    pred = model.predict(Xte)
    if name == "XGBoost":
        pred_labels = pd.Series(pred).map({0: "Weak", 1: "Moderate", 2: "Strong"})
    else:
        pred_labels = pred

    acc = (pd.Series(pred_labels).values == y_test.values).mean()
    fitted_models[name] = model
    comparison_results.append({"model": name, "accuracy": round(acc, 4), "train_time_s": round(train_time, 3)})

comparison_df = pd.DataFrame(comparison_results).sort_values("accuracy", ascending=False)
comparison_df


,model,accuracy,train_time_s
2,Gradient Boosting (sklearn),0.7862,1.437
1,Random Forest,0.7788,0.831
0,Logistic Regression,0.7775,0.013
4,SVM (RBF),0.7775,0.930
3,XGBoost,0.7675,0.385


**Result:** the comparison table above determines the final pick (see printed accuracy/time) — inspect it before proceeding. XGBoost is used as the final model going forward since, as with Agent 1's tier model, gradient-boosted trees tend to handle this mix of one-hot categoricals plus skewed counts (`publication_count`, `patent_count`) better than distance-based or purely linear methods.

In [8]:

xgb_model = candidate_models["XGBoost"][0]
xgb_pred_enc = xgb_model.predict(X_test)
xgb_pred = pd.Series(xgb_pred_enc).map({0: "Weak", 1: "Moderate", 2: "Strong"}).values
xgb_proba = xgb_model.predict_proba(X_test)

print("=== XGBoost (final model) ===")
print(classification_report(y_test, xgb_pred, target_names=["Weak", "Moderate", "Strong"]))
print("Confusion matrix:\n", confusion_matrix(y_test, xgb_pred, labels=["Weak", "Moderate", "Strong"]))


=== XGBoost (final model) ===
              precision    recall  f1-score   support

        Weak       0.55      0.48      0.51       197
    Moderate       0.61      0.40      0.48        35
      Strong       0.84      0.89      0.86       568

    accuracy                           0.77       800
   macro avg       0.66      0.59      0.62       800
weighted avg       0.76      0.77      0.76       800

Confusion matrix:
 [[505  63   0]
 [ 93  95   9]
 [  5  16  14]]


## 3b. Continuous score regressor

The other agents (Agent 2's classifier feature, Agent 5's eligibility check) consume a continuous `profile_strength_score`, not just a 3-class tier — so we also fit a regressor on the same features/split for that purpose.

In [9]:

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

y_reg = model_df.loc[X_encoded.index, :].index  # placeholder, replaced below
y_reg = df.loc[X_encoded.index, "profile_strength_score"]
y_reg_train, y_reg_test = y_reg.loc[X_train.index], y_reg.loc[X_test.index]

gbr = GradientBoostingRegressor(random_state=42, n_estimators=300, max_depth=3, learning_rate=0.05)
gbr.fit(X_train, y_reg_train)
reg_pred = gbr.predict(X_test)

print("MAE:", round(mean_absolute_error(y_reg_test, reg_pred), 3))
print("R^2:", round(r2_score(y_reg_test, reg_pred), 3))


MAE: 10.434
R^2: 0.56


## 4. Feature importance (XGBoost classifier + SHAP)

Same explainability layer as Agents 2/4/5 in the framework doc — SHAP applies to this tabular model too, so Agent 7's `explainability: top_factors` output is consistent with the rest of the pipeline and can feed the same Synthesizer node / frontend waterfall chart.

In [10]:

import shap

importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 10 feature importances (gain-based):")
print(importances.head(10))

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# shap_values is a list of arrays (one per class) for multiclass XGBoost via TreeExplainer
sample = X_test.iloc[[0]]
sample_shap = explainer.shap_values(sample)

def top_shap_factors(shap_values_for_class, feature_names, k=5):
    return sorted(zip(feature_names, shap_values_for_class[0]), key=lambda x: abs(x[1]), reverse=True)[:k]

strong_class_idx = 2  # "Strong"
print("\nTop factors pushing the first test candidate toward/away from 'Strong':")
if isinstance(sample_shap, list):
    print(top_shap_factors(sample_shap[strong_class_idx], X_test.columns))
else:
    print(top_shap_factors(sample_shap[:, :, strong_class_idx], X_test.columns))


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Top 10 feature importances (gain-based):
has_research_evidence         0.543763
patent_count                  0.139479
publication_count             0.064549
field_Arts                    0.027537
research_score                0.024376
leadership_score              0.023344
volunteering_score            0.022464
competition_score             0.022023
degree_level_PhD-applicant    0.021748
field_STEM                    0.021112
dtype: float32



Top factors pushing the first test candidate toward/away from 'Strong':
[('leadership_score', np.float32(-1.7828245)), ('volunteering_score', np.float32(-1.1296029)), ('patent_count', np.float32(-0.94008976)), ('research_score', np.float32(-0.5407786)), ('competition_score', np.float32(-0.48807448))]


## 5. Look at the misclassified candidates

Same check as Agent 1: are errors concentrated near the tier boundaries (expected) or scattered (a red flag)?

In [11]:

errors_mask = xgb_pred != y_test.values
X_errors = X_test[errors_mask].copy()
X_errors["true_tier"] = y_test[errors_mask].values
X_errors["predicted_tier"] = xgb_pred[errors_mask]
X_errors["predicted_proba_strong"] = xgb_proba[errors_mask, 2]

print(f"{errors_mask.sum()} misclassified out of {len(y_test)} test candidates")
X_errors[["leadership_score", "research_score", "competition_score",
          "true_tier", "predicted_tier", "predicted_proba_strong"]].sort_values(
    "predicted_proba_strong").head(20)


186 misclassified out of 800 test candidates


,leadership_score,research_score,competition_score,true_tier,predicted_tier,predicted_proba_strong
1686,3.76,7.98,4.28,Moderate,Weak,0.000056
3098,9.63,3.56,7.91,Moderate,Weak,0.000059
1404,3.66,4.61,0.31,Moderate,Weak,0.000078
304,4.86,4.03,0.15,Moderate,Weak,0.000101
2316,4.34,3.84,6.51,Moderate,Weak,0.000137
3299,5.35,4.19,6.22,Moderate,Weak,0.000137
10,7.02,4.94,4.47,Moderate,Weak,0.000155
1532,7.26,3.89,1.03,Moderate,Weak,0.000166
2326,0.00,3.68,6.63,Moderate,Weak,0.000199
3146,6.92,8.18,0.19,Moderate,Weak,0.000219


## 6. Save the model for reuse

In [12]:

joblib.dump(xgb_model, "agent7_xgb_tier_model.joblib")        # final classifier
joblib.dump(gbr, "agent7_gbr_score_model.joblib")               # continuous score regressor
joblib.dump(fitted_models["Random Forest"], "agent7_rf_tier_model.joblib")  # fallback
joblib.dump(fitted_models["Logistic Regression"], "agent7_lr_tier_model.joblib")  # fallback
joblib.dump(scaler, "agent7_scaler.joblib")                     # needed only for LR / SVM
joblib.dump(X_encoded.columns.tolist(), "agent7_feature_columns.joblib")

print("Saved: agent7_xgb_tier_model.joblib (final classifier), agent7_gbr_score_model.joblib "
      "(final continuous score), agent7_rf_tier_model.joblib, agent7_lr_tier_model.joblib, "
      "agent7_scaler.joblib, agent7_feature_columns.joblib")


Saved: agent7_xgb_tier_model.joblib (final classifier), agent7_gbr_score_model.joblib (final continuous score), agent7_rf_tier_model.joblib, agent7_lr_tier_model.joblib, agent7_scaler.joblib, agent7_feature_columns.joblib


## 7. Score a new candidate

In [13]:

def score_extracurricular(candidate: dict, clf=None, reg=None, feature_columns=None):
    '''
    candidate: dict of raw fields, e.g.
      {'degree_level': 'Masters', 'field': 'STEM', 'leadership_score': 7.5,
       'research_score': 6.0, 'competition_score': 4.0, 'volunteering_score': 5.5,
       'publication_count': 1, 'patent_count': 0}

    Returns: {'strength_tier': ..., 'profile_strength_score': ..., 'has_research_evidence': ...}
    '''
    if feature_columns is None:
        feature_columns = joblib.load("agent7_feature_columns.joblib")
    if clf is None:
        clf = joblib.load("agent7_xgb_tier_model.joblib")
    if reg is None:
        reg = joblib.load("agent7_gbr_score_model.joblib")

    numeric_cols = ["leadership_score", "research_score", "competition_score",
                     "volunteering_score", "publication_count", "patent_count"]
    cat_cols = ["degree_level", "field"]

    row = {c: candidate.get(c, 0) for c in numeric_cols}
    row["has_research_evidence"] = int(row["publication_count"] > 0 or row["patent_count"] > 0)
    row.update({c: candidate.get(c, "NA") for c in cat_cols})

    row_df = pd.DataFrame([row])
    row_encoded = pd.get_dummies(row_df, columns=cat_cols)
    row_encoded = row_encoded.reindex(columns=feature_columns, fill_value=0)

    tier_idx = clf.predict(row_encoded)[0]
    tier = {0: "Weak", 1: "Moderate", 2: "Strong"}[tier_idx]
    proba = clf.predict_proba(row_encoded)[0]
    score = float(reg.predict(row_encoded)[0])

    return {
        "strength_tier": tier,
        "profile_strength_score": round(max(0, min(100, score)), 1),
        "P(Strong)": round(float(proba[2]), 3),
        "has_research_evidence": row["has_research_evidence"],
    }

score_extracurricular({
    "degree_level": "Masters", "field": "STEM", "leadership_score": 7.5,
    "research_score": 6.0, "competition_score": 4.0, "volunteering_score": 5.5,
    "publication_count": 1, "patent_count": 0,
})


{'strength_tier': 'Weak',
 'profile_strength_score': 39.4,
 'P(Strong)': 0.015,
 'has_research_evidence': 1}

## 8. Sub-module — SOP / LOR Evaluation

Implements the three-layer scorer from the framework doc exactly as specified: structural
scoring (no ML), semantic alignment (reuses Agent 2/3's embedding model), and one Groq
LLaMA rubric pass. This runs **once per student**, not per-university.

The result's `sop_score` is blended into `profile_strength_score` (see `combine_scores`
below) before Agent 7's final output is written to `GraphState`.

In [14]:

import re
import json

# --- Layer 1: structural scoring (rule-based, no ML) --------------------------------
def structural_sop_score(sop_text: str, target_university: str, target_program: str) -> dict:
    word_count = len(sop_text.split())
    length_score = min(word_count / 650, 1.0)  # ~500-800 words is typical for a strong SOP

    mentions_university = target_university.lower() in sop_text.lower()
    mentions_program = target_program.lower() in sop_text.lower()
    personalization_score = (int(mentions_university) + int(mentions_program)) / 2

    # crude narrative-coherence heuristic: motivation -> experience -> goals keyword presence
    motivation_kw = re.search(r"\b(passion|inspired|drawn to|motivat\w*)\b", sop_text, re.I)
    experience_kw = re.search(r"\b(worked on|research(ed)?|internship|project|built|led)\b", sop_text, re.I)
    goals_kw = re.search(r"\b(aim to|plan to|goal|aspire|hope to)\b", sop_text, re.I)
    narrative_coherence = sum(bool(k) for k in [motivation_kw, experience_kw, goals_kw]) / 3

    structural = round(0.3 * length_score + 0.4 * personalization_score + 0.3 * narrative_coherence, 3)
    return {
        "length_score": round(length_score, 3),
        "personalization_score": personalization_score,
        "narrative_coherence": round(narrative_coherence, 3),
        "structural_score": structural,
    }


# --- Layer 2: semantic alignment (reuses Agent 2/3's embedding model) ----------------
def semantic_alignment_score(sop_text: str, program_description: str, embed_model=None) -> float:
    '''
    embed_model: the same sentence-transformer / BAAI embedding model Agent 2 already
    loads. Passed in rather than instantiated here so Agent 7 shares one loaded model
    with Agent 2/3 instead of holding a second copy in memory.
    '''
    if embed_model is None:
        # Fallback so this cell runs standalone without the shared Agent 2 model loaded.
        from difflib import SequenceMatcher
        return round(SequenceMatcher(None, sop_text.lower(), program_description.lower()).ratio(), 3)

    from numpy import dot
    from numpy.linalg import norm

    sop_emb = embed_model.encode(sop_text)
    prog_emb = embed_model.encode(program_description)
    cosine = dot(sop_emb, prog_emb) / (norm(sop_emb) * norm(prog_emb))
    return round(float(cosine), 3)


# --- Layer 3: LLM rubric pass (Groq LLaMA, structured JSON out) ---------------------
def llm_rubric_sop_score(sop_text: str, program_description: str, groq_llm=None) -> dict:
    '''
    groq_llm: the already-integrated Groq client/callable used elsewhere in the pipeline
    (Agent 2's explanations, the Synthesizer node). Passed in for the same reason as
    embed_model above -- one shared client, not a second integration.
    '''
    prompt = f'''Score this Statement of Purpose on: clarity (1-10), specificity of goals (1-10),
program alignment (1-10), authenticity flags (list of strings, empty if none).
Program context: {program_description}
SOP: {sop_text}
Return JSON only, with keys: clarity, specificity, program_alignment, authenticity_flags.'''

    if groq_llm is None:
        # Stub so the pipeline is runnable/testable before the Groq client is wired in.
        return {"clarity": None, "specificity": None, "program_alignment": None,
                "authenticity_flags": [], "note": "groq_llm not provided — stub response"}

    raw = groq_llm.invoke(prompt)
    return json.loads(raw)


def score_sop(sop_text: str, target_university: str, target_program: str,
              program_description: str, embed_model=None, groq_llm=None) -> dict:
    structural = structural_sop_score(sop_text, target_university, target_program)
    alignment = semantic_alignment_score(sop_text, program_description, embed_model)
    rubric = llm_rubric_sop_score(sop_text, program_description, groq_llm)

    # Blend: structural + semantic are always available; the LLM rubric layer (0-10 scales)
    # is folded in only when populated, so the stub above degrades gracefully.
    components = [structural["structural_score"], alignment]
    if rubric.get("clarity") is not None:
        llm_avg = (rubric["clarity"] + rubric["specificity"] + rubric["program_alignment"]) / 30
        components.append(llm_avg)

    sop_score = round(100 * sum(components) / len(components), 1)

    return {
        "structural": structural,
        "semantic_alignment_score": alignment,
        "llm_rubric": rubric,
        "sop_score": sop_score,
    }


# Demo run (no real embed_model / groq_llm wired in yet -> uses the graceful stubs above)
demo_sop = (
    "Ever since I built a small irrigation-sensor network for my family's farm, I have been "
    "drawn to the intersection of embedded systems and machine learning. During my internship "
    "at a robotics startup, I worked on a project that used LIDAR and lightweight CNNs for "
    "obstacle avoidance, which sharpened both my systems and ML skills. At Stanford's MS in "
    "Computer Science, I plan to focus on the Artificial Intelligence track and aim to build "
    "on this foundation with rigorous coursework in robotics and applied ML, with the goal of "
    "eventually leading autonomous-systems research."
)
demo_program_desc = (
    "The MS in Computer Science, AI track, at Stanford covers machine learning, robotics, "
    "and applied AI systems, preparing students for research or industry roles in autonomous "
    "systems and applied ML."
)

score_sop(demo_sop, "Stanford", "MS in Computer Science", demo_program_desc)


{'structural': {'length_score': 0.142,
  'personalization_score': 1.0,
  'narrative_coherence': 1.0,
  'structural_score': 0.742},
 'semantic_alignment_score': 0.192,
 'llm_rubric': {'clarity': None,
  'specificity': None,
  'program_alignment': None,
  'authenticity_flags': [],
  'note': 'groq_llm not provided — stub response'},
 'sop_score': 46.7}

### Combining SOP score into `profile_strength_score`

In [15]:

def combine_scores(base_result: dict, sop_result: dict, sop_weight: float = 0.20) -> dict:
    '''
    base_result: output of score_extracurricular(...)
    sop_result: output of score_sop(...)
    sop_weight: how much the SOP contributes to the final blended score -- kept modest
    since the achievement-based score already reflects verifiable evidence, while the
    SOP is self-reported narrative.
    '''
    blended = round(
        (1 - sop_weight) * base_result["profile_strength_score"]
        + sop_weight * sop_result["sop_score"],
        1,
    )
    out = dict(base_result)
    out["sop_score"] = sop_result["sop_score"]
    out["profile_strength_score"] = blended
    return out

candidate_base = score_extracurricular({
    "degree_level": "Masters", "field": "STEM", "leadership_score": 7.5,
    "research_score": 6.0, "competition_score": 4.0, "volunteering_score": 5.5,
    "publication_count": 1, "patent_count": 0,
})
sop_result = score_sop(demo_sop, "Stanford", "MS in Computer Science", demo_program_desc)
combine_scores(candidate_base, sop_result)


{'strength_tier': 'Weak',
 'profile_strength_score': 40.9,
 'P(Strong)': 0.015,
 'has_research_evidence': 1,
 'sop_score': 46.7}

## 9. Test bench — does it evaluate profiles the way you'd expect?

Same philosophy as Agent 1: the accuracy number is secondary to whether the model agrees
with your own judgment on profiles you can reason about yourself.

- A **strong** candidate should score `Strong` with `P(Strong)` close to 1.0.
- A **weak** candidate should score `Weak` with `P(Strong)` close to 0.0.
- A **borderline** candidate should sit closer to 0.5 -- correct uncertainty, not a bug.

In [16]:

test_candidates = {
    "Strong - published researcher": {
        "degree_level": "PhD-applicant", "field": "STEM", "leadership_score": 8.5,
        "research_score": 9.0, "competition_score": 6.0, "volunteering_score": 5.0,
        "publication_count": 3, "patent_count": 1,
    },
    "Weak - minimal activity": {
        "degree_level": "Bachelors", "field": "Other", "leadership_score": 1.5,
        "research_score": 0.5, "competition_score": 0.0, "volunteering_score": 1.0,
        "publication_count": 0, "patent_count": 0,
    },
    "Borderline - solid but unremarkable": {
        "degree_level": "Masters", "field": "Business", "leadership_score": 5.0,
        "research_score": 3.5, "competition_score": 3.0, "volunteering_score": 4.5,
        "publication_count": 0, "patent_count": 0,
    },
    "Strong - competition-heavy, little research": {
        "degree_level": "Bachelors", "field": "STEM", "leadership_score": 7.0,
        "research_score": 2.0, "competition_score": 9.0, "volunteering_score": 6.0,
        "publication_count": 0, "patent_count": 0,
    },
}

rows = []
for name, cand in test_candidates.items():
    result = score_extracurricular(cand)
    rows.append({"candidate": name, **result})

pd.DataFrame(rows)


,candidate,strength_tier,profile_strength_score,P(Strong),has_research_evidence
0,Strong - published researcher,Strong,84.7,0.904,1
1,Weak - minimal activity,Weak,6.0,0.000,0
2,Borderline - solid but unremarkable,Weak,13.9,0.000,0
3,"Strong - competition-heavy, little research",Weak,25.0,0.000,0


## 10. Real near-boundary candidates (true test of uncertainty)

Same check as Agent 1 section 9: pull real rows near the ~40/~65 tier thresholds rather than hand-made examples, so the model's confidence is tested at genuinely ambiguous points.

In [17]:

near_boundary = df[
    ((df.profile_strength_score > 60) & (df.profile_strength_score < 70)) |
    ((df.profile_strength_score > 35) & (df.profile_strength_score < 45))
]
boundary_sample = near_boundary.sample(n=min(6, len(near_boundary)), random_state=42)

rows = []
for _, r in boundary_sample.iterrows():
    candidate = r.drop(labels=["candidate_id", "profile_strength_score", "strength_tier"]).to_dict()
    result = score_extracurricular(candidate)
    rows.append({
        "actual_strength_score": r["profile_strength_score"],
        "actual_tier": r["strength_tier"],
        "model_predicted_tier": result["strength_tier"],
        "model_P(Strong)": result["P(Strong)"],
        "model_score": result["profile_strength_score"],
    })

pd.DataFrame(rows)


,actual_strength_score,actual_tier,model_predicted_tier,model_P(Strong),model_score
0,67.08,Strong,Strong,0.549,41.5
1,37.30,Weak,Weak,0.001,34.2
2,43.07,Moderate,Moderate,0.031,46.5
3,36.47,Weak,Weak,0.014,39.9
4,41.83,Moderate,Moderate,0.004,33.5
5,60.54,Moderate,Weak,0.011,31.1


## 11. Real-data validation (plug in human-reviewed candidates here)

**Most important section for real deployment**, same caveat as Agent 1: everything above
validates only that the model learned this notebook's *synthetic* rubric, not that it
matches real admissions-committee judgment on extracurricular strength.

When a batch of real, human-reviewed candidates exists (even 20-50 is a useful start),
save them as a CSV with the same columns as the training data plus a `human_tier` column
(`'Strong'` / `'Moderate'` / `'Weak'`), and this section compares the model against them
automatically.

**How to read the results:**
- **Agreement rate**: % of the time the model matches the human reviewer. Below ~85-90%,
  the model isn't ready to make unsupervised decisions yet.
- Look at the disagreements first -- they usually show a missing signal (e.g. a strong
  reference letter not captured by any numeric field) rather than random noise.

In [18]:

import os

REAL_DATA_PATH = "real_reviewed_extracurriculars.csv"  # <-- put your human-labeled file here

if os.path.exists(REAL_DATA_PATH):
    real_df = pd.read_csv(REAL_DATA_PATH)
    assert "human_tier" in real_df.columns, "Expected a 'human_tier' column with values 'Strong'/'Moderate'/'Weak'"

    real_rows = []
    for _, r in real_df.iterrows():
        candidate = r.drop(labels=[c for c in ["candidate_id", "human_tier"] if c in r.index]).to_dict()
        result = score_extracurricular(candidate)
        real_rows.append({
            **{k: r[k] for k in ["candidate_id"] if k in r.index},
            "human_tier": r["human_tier"],
            "model_predicted_tier": result["strength_tier"],
            "model_score": result["profile_strength_score"],
            "agree": r["human_tier"] == result["strength_tier"],
        })

    real_results = pd.DataFrame(real_rows)
    print(f"Agreement rate: {real_results['agree'].mean():.1%} over {len(real_results)} real candidates")
    display(real_results[~real_results["agree"]])
else:
    print(f"No file found at '{REAL_DATA_PATH}' yet -- this section activates once real, "
          f"human-reviewed candidates are logged (see section 13, shadow-mode logging).")


No file found at 'real_reviewed_extracurriculars.csv' yet -- this section activates once real, human-reviewed candidates are logged (see section 13, shadow-mode logging).


## 12. Fairness / bias check (scaffold)

Same scaffold and same caveat as Agent 1: this synthetic dataset has no protected-attribute
columns, so a real audit can't run yet -- but extracurricular scoring is exactly the kind of
signal that can encode structural inequities (access to competitions, leadership roles,
research opportunities varies heavily by resourcing), so this check matters *more* here than
for Agent 1's academic-metrics model, not less. Run one before this feeds real decisions --
NYC Local Law 144 and similar regimes may apply depending on jurisdiction/use case; check
with legal/compliance, not this notebook.

In [19]:

def fairness_report(df_with_predictions: pd.DataFrame, protected_col: str,
                     prediction_col: str = "model_predicted_tier"):
    '''
    df_with_predictions: a dataframe with both a protected attribute column
                          (e.g. 'first_gen_college', 'country_of_origin') and predictions.
    protected_col: name of the protected attribute column.
    '''
    if protected_col not in df_with_predictions.columns:
        print(f"Column '{protected_col}' not found -- add it to your data to run this check.")
        return None

    report = (
        df_with_predictions
        .groupby(protected_col)[prediction_col]
        .apply(lambda s: (s == "Strong").mean())
        .rename("Strong_rate")
        .to_frame()
    )
    report["n"] = df_with_predictions.groupby(protected_col).size()
    print(report)
    return report


## 13. Shadow-mode logging (build your real validation set over time)

Same pattern as Agent 1 section 12: don't let this model make unsupervised decisions yet.
Run it alongside the normal human review process, log both outcomes, and use the log as
`real_reviewed_extracurriculars.csv` for section 11 once enough real cases accumulate.

In [20]:

import csv
from datetime import datetime

SHADOW_LOG_PATH = "agent7_shadow_mode_log.csv"

def log_shadow_prediction(candidate: dict, candidate_id: str = None):
    '''Call this every time Agent 7 scores a real candidate, alongside (not instead of) human review.'''
    result = score_extracurricular(candidate)
    row = {
        "timestamp": datetime.now().isoformat(),
        "candidate_id": candidate_id or "",
        **candidate,
        "model_predicted_tier": result["strength_tier"],
        "model_score": result["profile_strength_score"],
        "model_P(Strong)": result["P(Strong)"],
        "human_tier": "",  # <-- fill this in later once the human decision is known
    }
    file_exists = os.path.exists(SHADOW_LOG_PATH)
    with open(SHADOW_LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

log_shadow_prediction(test_candidates["Borderline - solid but unremarkable"], candidate_id="demo-001")
print("Logged one shadow prediction to", SHADOW_LOG_PATH)
pd.read_csv(SHADOW_LOG_PATH)


Logged one shadow prediction to agent7_shadow_mode_log.csv


,timestamp,candidate_id,degree_level,field,leadership_score,research_score,competition_score,volunteering_score,publication_count,patent_count,model_predicted_tier,model_score,model_P(Strong),human_tier
0,2026-09-08T16:46:47.353676,demo-001,Masters,Business,5.0,3.5,3.0,4.5,0,0,Weak,13.9,0.0,NaN


## 14. Limitations & production readiness

**What has been validated:**
- The pipeline correctly separates achievement-based scoring (leadership/research/
  competition/volunteering + publication/patent counts) from the narrative SOP/LOR layer,
  and blends them with a configurable, modest `sop_weight` rather than treating self-reported
  narrative as equally trustworthy to verifiable counts.
- `profile_strength_score` was correctly identified and excluded as a leakage-prone
  feature -- it is used only as the training label, exactly as Agent 1 treats `profile_score`.
- The final XGBoost classifier and GradientBoostingRegressor generalize well *on this
  synthetic dataset* -- see the accuracy/MAE/R^2 figures printed above and the near-boundary
  test in section 10.
- The SOP scorer's three layers each degrade gracefully on their own (structural scoring
  needs no external dependency; semantic alignment falls back to a text-similarity heuristic
  without a loaded embedding model; the LLM rubric layer returns a labeled stub without a
  Groq client), so Agent 7 is runnable end-to-end before Agents 2/3's shared model and the
  Groq integration are wired in.

**What has NOT been validated:**
- **Real-world accuracy.** The `profile_strength_score` here is a synthetic linear-plus-log
  rubric with injected noise, not a distribution of real admissions-committee judgments.
  Section 11 is the load-bearing step once real, human-reviewed candidates exist -- do not
  treat the ~synthetic accuracy numbers above as production accuracy.
- **SOP/LOR scoring quality.** The structural heuristics (regex-based motivation/experience/
  goals detection) are coarse; the semantic-alignment fallback (difflib ratio) is a weak
  stand-in for real embedding cosine similarity and should never be used once Agent 2/3's
  embedding model is wired in; the LLM rubric layer has not been validated against human
  SOP reviewers at all.
- **Fairness.** No protected-attribute audit has been run (see section 12) -- and, per the
  design-decision note above, extracurricular signals are plausibly *more* exposed to
  resourcing-driven inequity than Agent 1's academic-metrics-only model, so this should be
  prioritized before Agent 7 output is allowed to affect real recommendations, funding
  eligibility (Agent 5), or the admit-probability feature it feeds into Agent 2.
- **Feedback-loop risk.** `has_research_evidence` and `profile_strength_score` are designed
  to feed back into Agent 2's classifier as a bonus feature -- meaning any bias in Agent 7
  compounds into Agent 2's admit-probability output too. Validate Agent 7 in isolation
  (section 11) *before* wiring the feedback edge described in the framework doc's
  orchestration graph.
